# 11 Karnataka hospital access equity

Question: how far does each person in Karnataka live from the nearest hospital, which districts
are worst off, and how unequal is access?

Inputs: S16 WorldPop 2025 1 km population (CC BY 4.0), S18 OpenStreetMap hospitals from the
owner's Geofabrik extract (ODbL), S17 district boundaries. Method: `src/atlas/analysis/access.py`
(exact great-circle nearest site, people-weighted statistics, weighted Gini and Lorenz curve).

Read every number with the limits in `summary.json`: straight-line distance, not travel time;
OSM hospitals are thinner in rural areas (distances there are upper bounds); hospitals of any
size count the same; WorldPop is a model.


In [ ]:
import json
import os
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from atlas.analysis.access import build

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/11_ka_hospital_access"))
result = build(OUT)
summary = result["summary"]
print(json.dumps(summary["statewide"], indent=1))

In [ ]:
lonlat, km, people = result["lonlat"], result["km"], result["people"]
fig, ax = plt.subplots(figsize=(7, 8))
sc = ax.scatter(
    lonlat[:, 0],
    lonlat[:, 1],
    c=np.minimum(km, 30),
    s=0.6,
    cmap="magma_r",
    marker="s",
    linewidths=0,
)
fig.colorbar(sc, ax=ax, label="km to nearest OSM hospital (capped at 30)")
ax.set_aspect("equal")
ax.set_title("Karnataka: straight-line distance to the nearest hospital, 1 km cells")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
fig.text(
    0.01,
    0.01,
    "S16 WorldPop 2025 (CC BY 4.0), S18 OpenStreetMap hospitals (ODbL). "
    "Straight-line; OSM coverage varies.",
    fontsize=6,
)
fig.savefig(OUT / "access_map.png", dpi=80, metadata={"Software": None})
plt.close(fig)

In [ ]:
curve = np.array(summary["lorenz"])
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(curve[:, 0], curve[:, 1], label=f"distance (Gini {summary['statewide']['gini']:.2f})")
ax.plot([0, 1], [0, 1], "--", color="grey", label="equal access")
ax.set_xlabel("share of people, nearest first")
ax.set_ylabel("share of total person-km")
ax.set_title("Lorenz curve of distance to a hospital")
ax.legend()
fig.savefig(OUT / "lorenz.png", dpi=80, metadata={"Software": None})
plt.close(fig)

In [ ]:
rows = result["rows"]
fig, ax = plt.subplots(figsize=(8, 7))
names = [r["district"] for r in rows][::-1]
ax.barh(names, [r["share_beyond_10km"] * 100 for r in rows][::-1], color="#b3261e")
ax.set_xlabel("% of people more than 10 km from a hospital (straight line)")
ax.set_title("Karnataka districts, farthest first")
fig.tight_layout()
fig.savefig(OUT / "district_beyond_10km.png", dpi=80, metadata={"Software": None})
plt.close(fig)
print(rows[:3])

## Limits

See `summary.json` field `limits`. This is an access-gap screen, not a planning standard:
a cell far from an OSM hospital may still have an unmapped clinic or primary health centre.